# Train SVM on Perch v2 embeddings

In [2]:
!pip install -q optuna 2>/dev/null

In [3]:
import os
os.environ['OMP_NUM_THREADS']      = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS']      = '1'

import time, warnings, joblib
import numpy as np
import pandas as pd
import optuna
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.INFO)
print('Imports OK')

Imports OK


In [4]:
COMPETITION_DATA_ROOT = '/kaggle/input/competitions/birdclef-2026'
PERCH_FEATURES_PATH   = '/kaggle/input/datasets/marialuizalaranjeira/perch-features/features_v4_perch_v2_onnx.parquet'
PERCH_LABELS_PATH     = None

OUTPUT_DIR  = '/kaggle/working'
RANDOM_SEED = 42

SKIP_HPO    = False
BEST_PARAMS = {'C': 0.001767, 'class_weight': 'balanced'}  

HPO_TIMEOUT_S     = 1 * 3600
HPO_TRAIN_FRAC    = 0.25
HPO_MAX_SPECIES   = 60
HPO_MIN_POS_TRAIN = 10
HPO_MIN_POS_VAL   = 3

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
class SafeClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimator):
        self.base_estimator = base_estimator
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        try:
            if len(self.classes_) < 2:
                raise ValueError('only one class')
            self.model_ = clone(self.base_estimator)
            self.model_.fit(X, y)
        except Exception:
            self.model_ = DummyClassifier(strategy='constant', constant=y[0])
            self.model_.fit(X, y)
        return self
    def predict_proba(self, X):
        m = self.model_
        if hasattr(m, 'predict_proba'):
            return m.predict_proba(X)
        d = m.decision_function(X)
        p = 1.0 / (1.0 + np.exp(-d))
        return np.column_stack([1.0 - p, p])

## Inspect the parquet

In [6]:
df = pd.read_parquet(PERCH_FEATURES_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()[:10]} ... (total {len(df.columns)})')
print(f'Dtypes (first 10):')
print(df.dtypes.head(10))
print('\nHead:')
print(df.head(3))

has_filename       = 'filename' in df.columns
has_all_labels     = 'all_labels' in df.columns
has_primary_label  = 'primary_label' in df.columns
emb_columns        = [c for c in df.columns if c.startswith('emb_') or c.startswith('embedding_')]
has_emb_array      = 'embedding' in df.columns and isinstance(df['embedding'].iloc[0], (list, np.ndarray))
print(f'\nhas_filename: {has_filename}')
print(f'has_all_labels (multi-label list): {has_all_labels}')
print(f'has_primary_label: {has_primary_label}')
print(f'embedding columns found: {len(emb_columns)}')
print(f'has_emb_array (single array col): {has_emb_array}')

Shape: (60808, 1540)
Columns: ['filename', 'start_sec', 'site', 'hour', 'perch_0', 'perch_1', 'perch_2', 'perch_3', 'perch_4', 'perch_5'] ... (total 1540)
Dtypes (first 10):
filename      object
start_sec    float64
site          object
hour           int64
perch_0      float32
perch_1      float32
perch_2      float32
perch_3      float32
perch_4      float32
perch_5      float32
dtype: object

Head:
       filename  start_sec     site  hour   perch_0   perch_1   perch_2  \
0  XC999239.ogg       15.0  unknown    -1 -0.181325 -0.017400 -0.029917   
1  XC999239.ogg       17.5  unknown    -1 -0.184825 -0.023360 -0.055797   
2  XC999239.ogg       12.5  unknown    -1 -0.172467 -0.025604 -0.006978   

    perch_3   perch_4   perch_5  ...  perch_1526  perch_1527  perch_1528  \
0 -0.033911  0.046721  0.041532  ...   -0.163781    0.068869    0.136497   
1  0.003495  0.046686 -0.061376  ...   -0.167418    0.116406    0.097085   
2 -0.065693  0.062313  0.017853  ...   -0.142637    0.068964    0.

In [7]:
import ast, os

taxonomy_df = pd.read_csv(f'{COMPETITION_DATA_ROOT}/taxonomy.csv')
ALL_SPECIES = taxonomy_df['primary_label'].unique().tolist()

perch_columns = sorted(
    [c for c in df.columns if c.startswith('perch_')],
    key=lambda c: int(c.split('_')[1]),
)
if not perch_columns:
    raise RuntimeError('No perch_* columns found — check the parquet.')
X_matrix = df[perch_columns].to_numpy(dtype=np.float32)
print(f'Feature matrix shape: {X_matrix.shape}  (using {len(perch_columns)} perch_* columns)')

groups = df['filename'].values

train_csv = pd.read_csv(f'{COMPETITION_DATA_ROOT}/train.csv')
train_csv['basename'] = train_csv['filename'].apply(os.path.basename)

def parse_secondary(s):
    if pd.isna(s) or s == '' or s == '[]':
        return []
    try:
        return list(ast.literal_eval(s))
    except Exception:
        return []

train_csv['all_labels'] = train_csv.apply(
    lambda r: [r['primary_label']] + parse_secondary(r['secondary_labels']),
    axis=1,
)
basename_to_labels = dict(zip(train_csv['basename'], train_csv['all_labels']))

parquet_basenames = df['filename'].astype(str).apply(os.path.basename).values
label_lists = [basename_to_labels.get(bn, []) for bn in parquet_basenames]

missing = sum(1 for L in label_lists if len(L) == 0)
print(f'Rows with no matching train.csv entry: {missing} / {len(label_lists)}')
if missing > 0:
    sample_missing = [bn for bn, L in zip(parquet_basenames, label_lists) if len(L) == 0][:5]
    print(f'  Sample unmatched basenames: {sample_missing}')
    print(f'  Sample train.csv basenames: {list(basename_to_labels.keys())[:5]}')

mlb = MultiLabelBinarizer(classes=ALL_SPECIES)
y_matrix_full = mlb.fit_transform(label_lists)
print(f'Targets shape: {y_matrix_full.shape}')
print(f'Total positive labels: {y_matrix_full.sum():,}')

Feature matrix shape: (60808, 1536)  (using 1536 perch_* columns)
Rows with no matching train.csv entry: 0 / 60808
Targets shape: (60808, 234)
Total positive labels: 81,469


In [8]:
counts        = y_matrix_full.sum(axis=0)
keep_indices  = np.where(counts > 0)[0]
ACTIVE_SPECIES = [ALL_SPECIES[i] for i in keep_indices]
y_matrix      = y_matrix_full[:, keep_indices]

print(f'Dropped {len(ALL_SPECIES) - len(ACTIVE_SPECIES)} zero-sample species.')
print(f'Training on {len(ACTIVE_SPECIES)} species.')

Dropped 49 zero-sample species.
Training on 185 species.


In [9]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_matrix)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, val_idx = next(gss.split(X_scaled, y_matrix, groups))

rng              = np.random.default_rng(RANDOM_SEED)
hpo_train_subset = rng.choice(train_idx,
                              size=int(len(train_idx) * HPO_TRAIN_FRAC),
                              replace=False)

X_hpo_train = X_scaled[hpo_train_subset]
y_hpo_train = y_matrix[hpo_train_subset]
X_hpo_val   = X_scaled[val_idx]
y_hpo_val   = y_matrix[val_idx]

candidates = [
    j for j in range(y_matrix.shape[1])
    if y_hpo_train[:, j].sum() >= HPO_MIN_POS_TRAIN
    and y_hpo_val[:, j].sum()   >= HPO_MIN_POS_VAL
]
candidates.sort(key=lambda j: y_hpo_train[:, j].sum(), reverse=True)
species_to_score = candidates[:HPO_MAX_SPECIES]

print(f'HPO train: {X_hpo_train.shape[0]:,}  val: {X_hpo_val.shape[0]:,}')
print(f'Species scored in HPO: {len(species_to_score)} (of {len(candidates)} eligible)')

HPO train: 12,166  val: 12,141
Species scored in HPO: 60 (of 158 eligible)


In [10]:
def objective(trial):
    C            = trial.suggest_float('C', 1e-5, 1e1, log=True)
    class_weight = trial.suggest_categorical('class_weight', ['balanced', None])

    aucs = []
    for j in species_to_score:
        try:
            clf = LogisticRegression(
                C=C, class_weight=class_weight,
                solver='liblinear',     
                max_iter=200,
                random_state=RANDOM_SEED,
            )
            clf.fit(X_hpo_train, y_hpo_train[:, j])
            scores = clf.decision_function(X_hpo_val)
            aucs.append(roc_auc_score(y_hpo_val[:, j], scores))
        except Exception:
            continue
    return float(np.mean(aucs)) if aucs else 0.0

In [ ]:
if SKIP_HPO:
    best = dict(BEST_PARAMS)
    print(f'Skipping HPO — using known-good params: {best}')
else:
    STUDY_DB_PATH = os.path.join(OUTPUT_DIR, 'optuna_study_perch.db')
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
        study_name='svm_birdclef_perch',
        storage=f'sqlite:///{STUDY_DB_PATH}',
        load_if_exists=True,
    )
    def progress_cb(study, trial):
        print(f'Trial {trial.number:3d}  value={trial.value:.4f}  '
              f'best={study.best_value:.4f}  params={trial.params}')
    t0 = time.time()
    study.optimize(objective, timeout=HPO_TIMEOUT_S, callbacks=[progress_cb])
    print(f'\nHPO finished in {(time.time()-t0)/60:.1f} min, {len(study.trials)} trials')
    print(f'Best value : {study.best_value:.4f}')
    best = study.best_params

print(f'Final params: {best}')

[I 2026-05-20 09:50:56,609] A new study created in RDB with name: svm_birdclef_perch
[I 2026-05-20 09:56:42,415] Trial 0 finished with value: 0.9260074505742525 and parameters: {'C': 0.0017670169402947942, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.9260074505742525.


Trial   0  value=0.9260  best=0.9260  params={'C': 0.0017670169402947942, 'class_weight': 'balanced'}


In [ ]:
base_estimator = LinearSVC(
    C=best['C'],
    class_weight=best['class_weight'],
    dual=False,
    max_iter=5000,
    tol=1e-3,
    random_state=RANDOM_SEED,
)
print(f'LinearSVC  C={best["C"]:.4g}  class_weight={best["class_weight"]}')

In [ ]:
print(f'Final fit on {X_scaled.shape[0]:,} samples × {X_scaled.shape[1]} features × {y_matrix.shape[1]} species')
print('Expect ~20–40 min (185 single LogReg fits, parallel across species).')
t0 = time.time()
final_model = OneVsRestClassifier(SafeClassifier(base_estimator), n_jobs=-1)
final_model.fit(X_scaled, y_matrix)
print(f'\nFinal training done in {(time.time() - t0)/60:.1f} min')

In [ ]:
model_path   = os.path.join(OUTPUT_DIR, 'svm_perch_model.pkl')
scaler_path  = os.path.join(OUTPUT_DIR, 'svm_perch_scaler.pkl')
species_path = os.path.join(OUTPUT_DIR, 'svm_perch_active_species.pkl')
params_path  = os.path.join(OUTPUT_DIR, 'svm_perch_best_params.pkl')

joblib.dump(final_model,    model_path)
joblib.dump(scaler,         scaler_path)
joblib.dump(ACTIVE_SPECIES, species_path)
joblib.dump(best,           params_path)

for p in (model_path, scaler_path, species_path, params_path):
    print(f'  {p}  ({os.path.getsize(p) / (1024*1024):.1f} MB)')